# K-ABENA Ch.14 — CIFAR-10 avec TensorFlow/Keras

**Miroir TensorFlow du notebook `01_cifar10_pytorch.ipynb`**
Même architecture ResNet-18, mêmes hyperparamètres, même protocole.

| Opération | PyTorch | TensorFlow |
|-----------|---------|------------|
| Filtre K-ABENA | `kabena_filter_torch(losses, K, N)` | `KabenaCallback(K, N)` |
| Loss individuelle | `reduction='none'` | automatique dans callback |
| Coût adoption | **+2 lignes** | **+1 callback** |

In [ ]:
# !pip install kabena-ml[tf] -q
import tensorflow as tf
import numpy as np
import time
from kabena_ml.core.filter import calibrate_K
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

EPOCHS = 10   # passer à 200 pour les résultats complets du ch.14
DEMO   = True
print(f'TF version: {tf.__version__} | Mode: {"DÉMO" if DEMO else "COMPLET"}')

In [ ]:
# ── Données CIFAR-10 ──────────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalisation identique au notebook PyTorch
MEAN = np.array([0.4914, 0.4822, 0.4465])
STD  = np.array([0.2023, 0.1994, 0.2010])
X_train = ((X_train.astype('float32') / 255.0) - MEAN) / STD
X_test  = ((X_test.astype('float32') / 255.0) - MEAN) / STD
y_train, y_test = y_train.squeeze(), y_test.squeeze()

# Augmentation (identique : RandomCrop + HorizontalFlip)
def augment(x, y):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.pad_to_bounding_box(x, 4, 4, 40, 40)  # padding 4
    x = tf.image.random_crop(x, [32, 32, 3])            # crop 32x32
    return x, y

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(50000).batch(128).map(augment).prefetch(2))
test_ds  = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(256).prefetch(2)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# ── Modèle ResNet-18 adapté CIFAR-10 ─────────────────────────────────────
def residual_block(x, filters, stride=1):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, 3, stride, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(filters, 3, 1, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, stride, use_bias=False)(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    return tf.keras.layers.ReLU()(x + shortcut)

def build_resnet18_cifar(n_classes=10):
    """ResNet-18 adapté CIFAR : conv3×3 stride=1 sans maxpool (identique au notebook PyTorch)."""
    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Conv2D(64, 3, 1, padding='same', use_bias=False)(inputs)  # stride=1
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    # Pas de MaxPool (identique à l'adaptation PyTorch)
    for filters, n_blocks, stride in [(64,2,1),(128,2,2),(256,2,2),(512,2,2)]:
        x = residual_block(x, filters, stride)
        for _ in range(n_blocks - 1):
            x = residual_block(x, filters)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(n_classes)(x)
    return tf.keras.Model(inputs, outputs)

print('Architecture ResNet-18 CIFAR : OK')

In [ ]:
# ── Expérience 1 : SGD standard (référence) ───────────────────────────────
print('='*60)
print('EXPÉRIENCE 1 — SGD standard')
print('='*60)

tf.random.set_seed(42); np.random.seed(42)
model_std = build_resnet18_cifar()
lr_sched  = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.1, decay_steps=EPOCHS*391)
model_std.compile(
    optimizer=tf.keras.optimizers.SGD(lr_sched, momentum=0.9, weight_decay=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
t0 = time.time()
hist_std = model_std.fit(train_ds, epochs=EPOCHS, validation_data=test_ds, verbose=1)
t_std = (time.time()-t0) / EPOCHS
acc_std = hist_std.history['val_accuracy'][-1] * 100
print(f'\n[Standard] Accuracy finale: {acc_std:.2f}% | {t_std:.1f}s/ép')

In [ ]:
# ── Expérience 2 : K-ABENA N=0.4 via KabenaCallback ─────────────────────
# SEUL CHANGEMENT vs standard : callbacks=[KabenaCallback(K=K_auto, N=0.4)]
print('='*60)
print('EXPÉRIENCE 2 — K-ABENA N=0.4 (KabenaCallback)')
print('Coût adoption : +1 callback dans model.fit()')
print('='*60)

# Calibrage auto de K sur les pertes de la première époque
model_temp = build_resnet18_cifar()
model_temp.compile(
    optimizer=tf.keras.optimizers.SGD(0.1, momentum=0.9),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)
losses_ep1 = []
for X_b, y_b in train_ds.take(5):  # 5 batchs pour calibrer
    logits = model_temp(X_b, training=False)
    l = tf.keras.losses.sparse_categorical_crossentropy(y_b, logits, from_logits=True)
    losses_ep1.extend(l.numpy())
K_auto = calibrate_K(np.array(losses_ep1), target_pct=0.10)
print(f'  K calibré automatiquement = {K_auto:.4f}')

tf.random.set_seed(42); np.random.seed(42)
model_ka4 = build_resnet18_cifar()
lr_sched2 = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.1, decay_steps=EPOCHS*391)
model_ka4.compile(
    optimizer=tf.keras.optimizers.SGD(lr_sched2, momentum=0.9, weight_decay=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

ka_cb = KabenaCallback(K=K_auto, N=0.4, verbose=True)  # ← SEUL AJOUT
t0 = time.time()
hist_ka4 = model_ka4.fit(
    train_ds, epochs=EPOCHS,
    validation_data=test_ds,
    callbacks=[ka_cb],   # ← +1 callback
    verbose=1
)
t_ka4 = (time.time()-t0) / EPOCHS
acc_ka4 = hist_ka4.history['val_accuracy'][-1] * 100
print(f'\n[K-ABENA N=0.4] Accuracy: {acc_ka4:.2f}% | {t_ka4:.1f}s/ép | {ka_cb.stats_summary()}')

In [ ]:
# ── Expérience 3 : K-ABENA Adaptatif (GradientTape) ──────────────────────
print('='*60)
print('EXPÉRIENCE 3 — K-ABENA Adaptatif (warm-up, GradientTape)')
print('='*60)

tf.random.set_seed(42); np.random.seed(42)
model_adp = build_resnet18_cifar()
cfg_adp   = KabenaConfig(K=K_auto, N=0.3, task='classification', verbose=True)
trainer   = KabenaTFTrainer(
    model_adp, cfg_adp,
    optimizer=tf.keras.optimizers.SGD(
        tf.keras.optimizers.schedules.CosineDecay(0.1, EPOCHS*391),
        momentum=0.9, weight_decay=1e-4
    )
)

# Warm-up adaptatif intégré
q_init, q_target, T_warmup = 5, 20, 20
t0 = time.time()
for epoch in range(EPOCHS):
    q   = q_init + (q_target - q_init) * min(epoch / T_warmup, 1)
    all_losses = []
    for X_b, y_b in train_ds:
        logits = model_adp(X_b, training=False)
        l = tf.keras.losses.sparse_categorical_crossentropy(y_b, logits, from_logits=True)
        all_losses.extend(l.numpy())
    K_t = float(np.percentile(all_losses, q))
    trainer.cfg.K = K_t  # update K adaptatif

    epoch_hist = trainer.fit(
        tf.constant(X_train), tf.constant(y_train),
        epochs=1, batch_size=128
    )
    val_logits = model_adp(X_test, training=False)
    val_acc    = (tf.argmax(val_logits, 1).numpy() == y_test).mean() * 100
    gain       = epoch_hist[-1]['gain_pct'] if epoch_hist else 0
    print(f'  Ép {epoch+1:3d} | K={K_t:.4f} (q={q:.0f}%) | acc={val_acc:.2f}% | gain={gain}%')

t_adp = (time.time()-t0) / EPOCHS
print(f'\n[K-ABENA Adaptatif TF] Accuracy: {val_acc:.2f}% | {t_adp:.1f}s/ép')

In [ ]:
# ── Tableau comparatif final ──────────────────────────────────────────────
import pandas as pd

results = [
    {'Variante': 'SGD standard (TF)',       'Acc.': f'{acc_std:.2f}%', 'Cible': '93.2%', 'Gain': '0%',   'Temps': f'{t_std:.1f}s'},
    {'Variante': 'K-ABENA N=0.4 TF',       'Acc.': f'{acc_ka4:.2f}%', 'Cible': '94.6%', 'Gain': '14.2%','Temps': f'{t_ka4:.1f}s'},
    {'Variante': 'K-ABENA Adaptatif TF',   'Acc.': f'{val_acc:.2f}%', 'Cible': '94.9%', 'Gain': '19.3%','Temps': f'{t_adp:.1f}s'},
]
print('\n' + '='*65)
print('RÉSULTATS CIFAR-10 (TensorFlow) — Comparaison avec Ch.14')
print('='*65)
print(pd.DataFrame(results).to_string(index=False))
print('\nGuide migration :')
print('  PyTorch  → kabena_filter_torch(losses, K, N)  (+2 lignes)')
print('  TF Keras → KabenaCallback(K, N)               (+1 callback)')
print('  TF Tape  → KabenaTFTrainer + trainer.filter() (+2 lignes)')